# 08 — Pricing the TI derivative markets (Radiant/Dire · Longest game · Most banned)

These three books are where match data beats the crowd: the outcome is a
public statistic, so a model of the data-generating process competes
against a thin retail book rather than against sharp bettors.

**Discipline:** each model's answer is dominated by one input (the true
current-patch radiant rate; the duration tail; the ban prior). Always run
the sensitivity view before sizing — a point estimate here is a lie.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)
DATA = ROOT / 'data'


In [ ]:
from src.derivatives import (radiant_dire_market, radiant_dire_sensitivity,
                             longest_game_market, most_banned_hero_market,
                             compare_to_book)
from src.polymarket import all_ti_books
books = all_ti_books()
for k, b in books.items():
    print(f"\n=== {k}: {b.attrs['title']} | vol ${b.attrs['volume']:,.0f} | "
          f"priced-sum {b.attrs['priced_sum']:.3f} bid-sum {b.attrs['bid_sum']:.3f}")
    print(b.head(8).to_string(index=False))

## A. Radiant vs Dire
Set `PRIOR_P` from notebook 07's **current-patch** radiant rate, and
`GAMES_TOTAL` from the actual TI format. Update `PLAYED`/`RAD_WINS` daily
as the tournament progresses — every game shrinks the uncertainty.

In [ ]:
PRIOR_P     = 0.530   # <- current-patch pro radiant win rate (notebook 07)
PRIOR_K     = 400     # pseudo-games of confidence in that prior
PLAYED      = 0       # TI games completed so far
RAD_WINS    = 0       # of which won by Radiant
GAMES_LEFT  = 180     # remaining games in the tournament
r = radiant_dire_market(PLAYED, RAD_WINS, GAMES_LEFT, PRIOR_P, PRIOR_K)
print(r['fair_prices'], '| posterior p = %.4f +- %.4f' % (r['posterior_p_mean'], r['posterior_p_sd']))
print('\nSENSITIVITY  P(Radiant)  [rows = true p, cols = total games]')
print(radiant_dire_sensitivity().to_string())
book = dict(zip(books['radiant_dire'].outcome, books['radiant_dire'].mid))
model = pd.DataFrame({'outcome': list(r['fair_prices']), 'p': list(r['fair_prices'].values())})
compare_to_book(model, book, 'outcome', 'p')

## B. Longest single game
Feed a **same-patch** duration sample (hundreds of games). `empirical`
cannot produce a game longer than your sample's max — if the buckets sit
near/above that max, `gpd` is the only honest model.

In [ ]:
matches = pd.read_parquet(DATA / 'matches.parquet')
cur = matches.patch.max()
dur = (matches.loc[matches.patch == cur, 'duration'] / 60).dropna().to_numpy()
print(f'patch {cur}: n={len(dur)} median={np.median(dur):.1f} p99={np.quantile(dur,.99):.1f} max={dur.max():.1f}')
BUCKETS = [(91,95),(96,100),(101,105),(106,110),(111,None)]
LONGEST_SO_FAR = 0.0   # <- update from live TI results
for tm in ('empirical','gpd'):
    out = longest_game_market(dur, GAMES_LEFT, BUCKETS, LONGEST_SO_FAR, tail_model=tm)
    print(f"\n[{tm}] E[max]={out.attrs['sim_max_mean']:.1f} p95={out.attrs['sim_max_p95']:.0f}")
    print(out.to_string(index=False))

## C. Most banned hero
Dirichlet-multinomial forward simulation from bans so far + a same-patch
prior. The tie mass matters: the book resolves ties alphabetically, so
treat `p_tie_any` as a haircut on the leader.

In [ ]:
from src.draft import ban_table
pb     = pd.read_parquet(DATA / 'picks_bans.parquet')
heroes = pd.read_parquet(DATA / 'heroes.parquet') if (DATA/'heroes.parquet').exists() else None
# prior = ban shares from recent same-patch tier-1 play
prior_tbl = ban_table(pb[pb.match_id.isin(matches.loc[matches.patch==cur,'match_id'])], heroes)
prior_rates = prior_tbl.set_index('hero_id')['bans']
TI_BANS_SO_FAR = prior_rates * 0   # <- replace with live TI ban counts
res = most_banned_hero_market(TI_BANS_SO_FAR, games_played=0,
                              games_remaining=GAMES_LEFT, prior_rates=prior_rates)
print('tie mass:', round(res.attrs['p_tie_any'], 4), '| ban slots left:', res.attrs['remaining_ban_slots'])
if heroes is not None:
    nm = dict(zip(heroes.id, heroes.localized_name))
    res['hero'] = res.hero_id.map(nm)
res.head(15)

**Decision rule** (from `research/STRATEGY_SELECTION.md`): trade only where
the model edge survives fees + half-spread **and** the book is genuinely
thin (no bid = post a bid, never cross a 0.49 placeholder ask). Size with
fractional Kelly and treat all three markets as ONE cluster — they are
driven by the same tournament and correlate.